# Curtaining score — Fourier wedge method (step by step)

This notebook measures **curtaining** (the faint vertical streaks left by FIB milling)
in TIFF images and writes the results to a CSV. It follows the method used in
*Křepelka et al., "Automated cryo-volume EM"* (which  builds on *Dumoux et al.*):

> Keep only the frequencies inside a narrow angular **wedge** of the 2D Fourier
> transform, turn them back into an image (which then contains essentially only the
> curtains), and measure how strong that stripe image is.

For every image we save four pictures so you can see exactly what happened:
1. the **FFT** (frequency map) of the image,
2. the **wedge-thresholded FFT** (only the stripe frequencies kept),
3. the **spatially recovered** stripe-only image (inverse FFT of the wedge), and
4. a **heatmap of curtains** (local score per tile, since curtaining is patchy).

The wedge half-angle is a **parameter** (`WEDGE_ANGLE_DEG`), not fixed at 5°.

## Step 0 — Configuration

Edit these values and run the notebook top to bottom.

- **`INPUT_PATH`** — folder containing the `.tif` images to score (searched recursively).
- **`OUTPUT_PATH`** — where the CSV and the saved pictures go (defaults to `INPUT_PATH` if left blank).
- **`WEDGE_ANGLE_DEG`** — the wedge half-angle in degrees (the paper uses 5). Larger = catches
  more slanted / wavy curtains but also more real content; smaller = stricter.
- **`R0_FRAC` / `R1_FRAC`** — inner/outer radius of the frequency band as a fraction of the
  maximum radius. `R0_FRAC` excludes the DC / lowest frequencies (overall brightness), `R1_FRAC`
  is the outer limit (1.0 = keep everything out to the corners).
- **`USE_GRADIENT`** — if `True`, differentiate across the stripes (Sobel-x) before the FFT to
  boost stripe contrast. Default `False` to match the paper's plain-image definition.
- **`TILE`** — tile size (pixels) for the local curtaining heatmap.
- **`SAVE_PICTURES`** — save the FFT / wedge / recovered / heatmap PNGs per image.

In [ ]:
import os

import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.filters import threshold_li

# ------------------------------------------------------------------ config ---
INPUT_PATH  = r"../data/images"
OUTPUT_PATH = r"../outputs/curtaining"

WEDGE_ANGLE_DEG = 5.0     # wedge half-angle in degrees (paper uses 5)
R0_FRAC         = 0.02    # inner radius (excludes DC / brightness); fraction of r_max
R1_FRAC         = 1.0     # outer radius; fraction of r_max
USE_GRADIENT    = False   # Sobel-x front end before FFT (boosts stripe contrast)
TILE            = 256     # tile size (px) for the local heatmap
SAVE_PICTURES   = True    # save FFT / wedge / recovered / heatmap PNGs per image

# Restrict the analysis to the real imaged area (ignore the zero background that
# rotated/aligned frames leave behind). Only pixels strictly greater than
# MASK_THRESHOLD are used for the scores; the background is mean-filled before the
# FFT so its hard edge does not inject fake stripes.
USE_NONZERO_MASK = True   # if True, only use pixels > MASK_THRESHOLD
MASK_THRESHOLD   = 0.0    # pixels <= this value are treated as background
MIN_VALID_FRAC   = 0.5    # heatmap tiles with fewer valid pixels than this are skipped

CSV_NAME = "curtaining_wedge_scores.csv"

OUT_DIR = OUTPUT_PATH if OUTPUT_PATH else INPUT_PATH
os.makedirs(OUT_DIR, exist_ok=True)
PIC_DIR = os.path.join(OUT_DIR, "curtaining_diagnostics")
if SAVE_PICTURES:
    os.makedirs(PIC_DIR, exist_ok=True)

print(f"Input folder  : {INPUT_PATH}")
print(f"Output folder : {OUT_DIR}")
print(f"Wedge angle   : +/- {WEDGE_ANGLE_DEG} deg   |   radius band: {R0_FRAC}-{R1_FRAC} of r_max")
print(f"Gradient front end: {USE_GRADIENT}   |   heatmap tile: {TILE} px")
print(f"Non-zero mask : {USE_NONZERO_MASK} (pixels > {MASK_THRESHOLD})")

## Step 1 — Find and load the images

`find_tifs` walks the input folder and returns every `.tif`, skipping ilastik /
segmentation / mask companion files. `load_image` reads a TIFF as a single 2-D
grayscale image (channel 0 for colour images, the middle slice of a z-stack).

In [ ]:
def find_tifs(root, skip_tokens=("_Probabilities", "_Simple Segmentation", "_full_cell")):
    """Walk `root` and return sorted .tif files, skipping companion/mask files."""
    files = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if not fname.lower().endswith(".tif"):
                continue
            if any(tok in fname for tok in skip_tokens):
                continue
            files.append(os.path.join(dirpath, fname))
    return sorted(files)


def load_image(path):
    """Load a TIFF as a single 2-D float64 image.

    Multi-channel images (H, W, C) use channel 0; z-stacks use the middle slice.
    """
    img = np.asarray(tifffile.imread(path))
    if img.ndim == 3:
        if img.shape[-1] <= 4:          # (H, W, C) -> first channel
            img = img[..., 0]
        else:                            # (Z, H, W) -> middle slice
            img = img[img.shape[0] // 2]
    return img.astype(np.float64)


tif_files = find_tifs(INPUT_PATH)
print(f"Found {len(tif_files)} TIFF file(s) under {INPUT_PATH}")
for p in tif_files[:10]:
    print("  ", os.path.basename(p))
if len(tif_files) > 10:
    print(f"   ... and {len(tif_files) - 10} more")

## Step 2 — Fourier transform of the image

We subtract the mean (so the flat brightness doesn't dominate) and multiply by a 2-D
**Hann window** before the FFT. The window tapers the image to zero at its borders so
that the natural discontinuity between the left/right and top/bottom edges does not
create a fake bright cross in the FFT that could be mistaken for stripes.

`compute_fft` returns the complex, centred spectrum `F` (used to rebuild the image
later) and its power `|F|^2` (used for display and the energy ratio).

When `USE_NONZERO_MASK` is on, the zero background is first replaced with the mean of
the real (non-zero) pixels via `prep_for_fft`. A hard black border is itself a strong
straight edge and would add its own frequency streaks; filling it with the mean makes
it disappear after mean-subtraction and windowing, so only the true imaged area
contributes.

In [ ]:
def valid_mask(image, use_mask=USE_NONZERO_MASK, threshold=MASK_THRESHOLD):
    """Boolean mask of the real imaged area (pixels strictly greater than threshold)."""
    img = np.asarray(image)
    if not use_mask:
        return np.ones(img.shape, dtype=bool)
    return img > threshold


def prep_for_fft(image, valid):
    """Fill background (invalid) pixels with the mean of the valid pixels.

    A hard zero border is a straight edge that injects fake stripe frequencies; the
    mean fill removes ~all of it after mean-subtraction + windowing in compute_fft.
    """
    img = np.asarray(image, dtype=np.float64)
    if valid is None or valid.all() or not valid.any():
        return img
    out = img.copy()
    out[~valid] = img[valid].mean()
    return out


def compute_fft(image, use_gradient=False):
    """Return (F, power) where F is the centred complex spectrum and power = |F|^2.

    A Hann window suppresses border-wraparound artifacts. If `use_gradient` is set,
    a Sobel-x gradient (differentiating ACROSS the vertical stripes) is applied first
    to amplify the stripes relative to smooth content.
    """
    img = np.asarray(image, dtype=np.float64)
    if use_gradient:
        img = ndimage.sobel(img, axis=1, mode="nearest")   # axis=1 -> across x
    img = img - img.mean()

    ny, nx = img.shape
    window = np.outer(np.hanning(ny), np.hanning(nx))
    F = np.fft.fftshift(np.fft.fft2(img * window))
    power = np.abs(F) ** 2
    return F, power

## Step 3 — Build the wedge mask (angle is a parameter)

We measure, for every point in the FFT, its **angle away from the horizontal frequency
axis**. Points on that axis have angle `0`; the vertical axis has angle `90`. Vertical
stripes live near angle `0`, so the **wedge** keeps points with `angle <= WEDGE_ANGLE_DEG`
that also fall inside the radius band `R0_FRAC .. R1_FRAC` (the band excludes DC and the
extreme corners).

We also build the full **annulus** (same radius band, *all* angles). Comparing wedge
energy to annulus energy gives a contrast-invariant score later on.

In [ ]:
def _radius_angle(shape):
    """Per-pixel radius (0-1 of r_max) and angle-from-horizontal-axis (deg)."""
    ny, nx = shape
    cy, cx = ny // 2, nx // 2
    yy, xx = np.ogrid[:ny, :nx]
    dy, dx = yy - cy, xx - cx
    radius = np.sqrt(dx ** 2 + dy ** 2)
    rmax = np.sqrt(cx ** 2 + cy ** 2)
    radius = radius / rmax if rmax > 0 else radius
    # angle away from the horizontal (u) axis: 0 deg on the axis, 90 deg vertical
    angle = np.degrees(np.arctan2(np.abs(dy), np.abs(dx)))
    return radius, angle


def wedge_mask(shape, theta_deg=WEDGE_ANGLE_DEG, r0_frac=R0_FRAC, r1_frac=R1_FRAC):
    """Boolean mask: thin angular wedge about the horizontal frequency axis."""
    radius, angle = _radius_angle(shape)
    in_band = (radius >= r0_frac) & (radius <= r1_frac)
    return in_band & (angle <= theta_deg)


def annulus_mask(shape, r0_frac=R0_FRAC, r1_frac=R1_FRAC):
    """Boolean mask: full ring at the same radii, all angles (for normalisation)."""
    radius, _ = _radius_angle(shape)
    return (radius >= r0_frac) & (radius <= r1_frac)


# quick look at the wedge for a typical image size
_demo = wedge_mask((512, 512))
print(f"Wedge keeps {_demo.mean() * 100:.2f}% of the FFT at +/-{WEDGE_ANGLE_DEG} deg "
      f"(an isotropic image would put roughly this fraction of its energy in the wedge).")

## Step 4 — Recover the stripe-only image

Zero out everything in the spectrum except the wedge, then run the **inverse FFT**.
What comes back is an image that contains essentially *only the curtains* — this is the
"spatially recovered" image the notebook saves. It is the heart of the paper's method:
the curtaining score is simply how much this stripe-only image varies.

In [ ]:
def recover_stripes(F, mask):
    """Keep only the masked frequencies and inverse-transform to the stripe image."""
    Fw = np.zeros_like(F)
    Fw[mask] = F[mask]
    stripes = np.real(np.fft.ifft2(np.fft.ifftshift(Fw)))
    return stripes

## Step 5 — Threshold to a curtain mask, and compute the scores

All three numbers are measured **only over the valid pixels** (pixels `> MASK_THRESHOLD`,
i.e. the real imaged area) when `USE_NONZERO_MASK` is on, so the zero background never
contributes. From the stripe-only image we compute (all **higher = more curtaining**):

- **`curtaining_std`** — standard deviation of the stripe-only image. This is the
  paper's (Křepelka/Dumoux) definition. It scales with image contrast, so only compare
  it between images acquired the same way.
- **`wedge_energy_ratio`** — wedge energy ÷ annulus energy, on a 0–100 scale. This
  divides out overall contrast, so a clean isotropic image sits near the small wedge
  floor (see Step 3) regardless of brightness. The most robust single number.
- **`pct_curtain_pixels`** — we auto-threshold the magnitude of the stripe image with
  Li's minimum-cross-entropy method (the Dumoux step) and report the percentage of
  pixels flagged as curtain. This also gives us the binary **curtain mask** picture.

> **Read `pct_curtain_pixels` together with `wedge_energy_ratio`.** Auto-thresholding a
> near-flat stripe image (a genuinely clean picture) still splits roughly a third of the
> pixels, so the percentage on its own is only meaningful once real stripes are present.
> `wedge_energy_ratio` is the reliable stand-alone discriminator: it sits near the small
> wedge floor for clean images and climbs toward 100 as curtaining takes over.

In [ ]:
def threshold_curtains(stripes, valid=None):
    """Binary curtain mask + % curtain pixels via Li min-cross-entropy threshold.

    Only the valid (real imaged) pixels are used to pick the threshold and to compute
    the percentage, so the zero background is never counted.
    """
    mag = np.abs(stripes)
    if valid is None:
        valid = np.ones_like(mag, dtype=bool)
    vals = mag[valid]
    if vals.size == 0 or vals.max() <= vals.min():
        return np.zeros_like(mag, dtype=bool), 0.0
    try:
        t = threshold_li(vals)
    except Exception:
        t = vals.mean() + vals.std()
    mask = (mag > t) & valid
    return mask, 100.0 * float(mask.sum()) / float(valid.sum())


def analyze_image(image, theta_deg=WEDGE_ANGLE_DEG, r0_frac=R0_FRAC,
                  r1_frac=R1_FRAC, use_gradient=USE_GRADIENT,
                  use_mask=USE_NONZERO_MASK, threshold=MASK_THRESHOLD):
    """Full wedge analysis of one image.

    Returns the intermediate pictures (for saving) and the three scores. When
    `use_mask` is on, only pixels > `threshold` (the real imaged area) drive the
    scores, and the background is mean-filled before the FFT.
    """
    img = np.asarray(image, dtype=np.float64)
    valid = valid_mask(img, use_mask, threshold)
    img_f = prep_for_fft(img, valid)

    F, power = compute_fft(img_f, use_gradient=use_gradient)
    wmask = wedge_mask(img.shape, theta_deg, r0_frac, r1_frac)
    amask = annulus_mask(img.shape, r0_frac, r1_frac)

    stripes = recover_stripes(F, wmask)
    curtain_mask, pct = threshold_curtains(stripes, valid)

    annulus_energy = float(power[amask].sum())
    wedge_energy = float(power[wmask].sum())
    ratio = (wedge_energy / annulus_energy) if annulus_energy > 0 else np.nan

    std_score = float(np.std(stripes[valid])) if valid.any() else np.nan

    return {
        "log_power": np.log1p(power),                 # FFT picture
        "wedge_log_power": np.log1p(power * wmask),    # wedge-thresholded FFT picture
        "wedge_mask": wmask,
        "valid_mask": valid,
        "stripes": stripes,                            # spatially recovered picture
        "curtain_mask": curtain_mask,                  # thresholded curtain picture
        "curtaining_std": std_score,
        "wedge_energy_ratio": float(100.0 * ratio) if np.isfinite(ratio) else np.nan,
        "pct_curtain_pixels": pct,
        "valid_frac": float(valid.mean()),
    }

## Step 6 — Local curtaining heatmap

Curtaining is patchy — it can be strong in one band of the image and absent elsewhere.
We tile the image into `TILE`x`TILE` blocks and compute the contrast-invariant
**wedge energy ratio** for each tile. The resulting small grid is the "heatmap of
curtains": bright tiles = local curtaining. It is also the best way to spot a false
positive (a single hot tile over a real vertical feature vs. curtains spread across
many tiles).

In [ ]:
def local_heatmap(image, tile=TILE, theta_deg=WEDGE_ANGLE_DEG,
                  r0_frac=R0_FRAC, r1_frac=R1_FRAC, use_gradient=USE_GRADIENT,
                  use_mask=USE_NONZERO_MASK, threshold=MASK_THRESHOLD,
                  min_valid_frac=MIN_VALID_FRAC):
    """Per-tile wedge energy ratio (0-100). Returns a small 2-D grid.

    Tiles that are mostly background (valid fraction < min_valid_frac) are left NaN
    so the black border does not paint the heatmap.
    """
    img = np.asarray(image, dtype=np.float64)
    ny, nx = img.shape
    row_starts = list(range(0, ny, tile))
    col_starts = list(range(0, nx, tile))
    hmap = np.full((len(row_starts), len(col_starts)), np.nan)

    for i, y0 in enumerate(row_starts):
        for j, x0 in enumerate(col_starts):
            patch = img[y0:y0 + tile, x0:x0 + tile]
            if patch.shape[0] < 8 or patch.shape[1] < 8:
                continue
            if use_mask and (patch > threshold).mean() < min_valid_frac:
                continue
            res = analyze_image(patch, theta_deg, r0_frac, r1_frac, use_gradient,
                                use_mask, threshold)
            hmap[i, j] = res["wedge_energy_ratio"]
    return hmap

## Step 7 — Visualise and save the pictures for one image

`diagnostics` runs the analysis, builds a 6-panel figure (original, FFT,
wedge-thresholded FFT, recovered stripes, curtain mask, heatmap) and, when
`SAVE_PICTURES` is on, writes each panel as its own PNG under
`curtaining_diagnostics/` plus the combined panel. Run the next cell to preview the
first image in the folder.

In [ ]:
def _save_gray(array, out_png, title="", cmap="gray"):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(array, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(out_png, dpi=120)
    plt.close(fig)


def diagnostics(image, stem, out_dir=PIC_DIR, save=SAVE_PICTURES, show=True):
    """Analyse one image, draw a 6-panel figure, and optionally save every panel."""
    res = analyze_image(image)
    hmap = local_heatmap(image)
    res["heatmap"] = hmap

    if save:
        _save_gray(image, os.path.join(out_dir, stem + "_00_original.png"),
                   "original", cmap="gray")
        _save_gray(res["log_power"], os.path.join(out_dir, stem + "_01_fft.png"),
                   "FFT (log power)", cmap="viridis")
        _save_gray(res["wedge_log_power"], os.path.join(out_dir, stem + "_02_wedge_fft.png"),
                   f"wedge-thresholded FFT (+/-{WEDGE_ANGLE_DEG} deg)", cmap="viridis")
        _save_gray(res["stripes"], os.path.join(out_dir, stem + "_03_stripes.png"),
                   "recovered stripes", cmap="gray")
        _save_gray(res["curtain_mask"], os.path.join(out_dir, stem + "_04_curtain_mask.png"),
                   "curtain mask (Li threshold)", cmap="gray")
        _save_gray(hmap, os.path.join(out_dir, stem + "_05_heatmap.png"),
                   "local curtaining heatmap", cmap="magma")

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes[0, 0].imshow(image, cmap="gray");              axes[0, 0].set_title("original")
    axes[0, 1].imshow(res["log_power"], cmap="viridis"); axes[0, 1].set_title("FFT (log power)")
    axes[0, 2].imshow(res["wedge_log_power"], cmap="viridis")
    axes[0, 2].set_title(f"wedge-thresholded FFT (+/-{WEDGE_ANGLE_DEG} deg)")
    axes[1, 0].imshow(res["stripes"], cmap="gray");     axes[1, 0].set_title("recovered stripes")
    axes[1, 1].imshow(res["curtain_mask"], cmap="gray"); axes[1, 1].set_title("curtain mask")
    im = axes[1, 2].imshow(hmap, cmap="magma", vmin=0)
    axes[1, 2].set_title("local heatmap (wedge ratio)")
    fig.colorbar(im, ax=axes[1, 2], fraction=0.046)
    for ax in axes.ravel():
        ax.axis("off")
    fig.suptitle(
        f"{stem}   |   std={res['curtaining_std']:.3g}   "
        f"ratio={res['wedge_energy_ratio']:.2f}   curtain px={res['pct_curtain_pixels']:.2f}%",
        fontsize=13,
    )
    fig.tight_layout()
    if save:
        fig.savefig(os.path.join(out_dir, stem + "_panel.png"), dpi=120)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return res


if tif_files:
    _first = tif_files[0]
    _stem = os.path.splitext(os.path.basename(_first))[0]
    _res = diagnostics(load_image(_first), _stem)
    print(f"curtaining_std      = {_res['curtaining_std']:.4g}")
    print(f"wedge_energy_ratio  = {_res['wedge_energy_ratio']:.2f}  (0-100, higher = worse)")
    print(f"pct_curtain_pixels  = {_res['pct_curtain_pixels']:.2f} %")
else:
    print("No TIFF files found — check INPUT_PATH in Step 0.")

## Step 8 — Batch: score every image and write the CSV

Runs the analysis on every TIFF, saves the per-image pictures (when `SAVE_PICTURES` is
on), and writes one row per image to `curtaining_wedge_scores.csv` in the output folder.
The CSV columns are the three scores plus the paths to the saved pictures.

In [ ]:
rows = []
n = len(tif_files)
for k, path in enumerate(tif_files, 1):
    stem = os.path.splitext(os.path.basename(path))[0]
    try:
        img = load_image(path)
        res = diagnostics(img, stem, show=False)   # analyse + save pictures, no popup
        row = {
            "file": os.path.basename(path),
            "path": path,
            "curtaining_std": res["curtaining_std"],
            "wedge_energy_ratio": res["wedge_energy_ratio"],
            "pct_curtain_pixels": res["pct_curtain_pixels"],
            "valid_frac": res["valid_frac"],
            "wedge_angle_deg": WEDGE_ANGLE_DEG,
        }
        if SAVE_PICTURES:
            row["fft_png"]      = os.path.join(PIC_DIR, stem + "_01_fft.png")
            row["wedge_fft_png"] = os.path.join(PIC_DIR, stem + "_02_wedge_fft.png")
            row["stripes_png"]  = os.path.join(PIC_DIR, stem + "_03_stripes.png")
            row["curtain_mask_png"] = os.path.join(PIC_DIR, stem + "_04_curtain_mask.png")
            row["heatmap_png"]  = os.path.join(PIC_DIR, stem + "_05_heatmap.png")
            row["panel_png"]    = os.path.join(PIC_DIR, stem + "_panel.png")
        rows.append(row)
        print(f"[{k}/{n}] {os.path.basename(path):40s} "
              f"std={res['curtaining_std']:.3g}  ratio={res['wedge_energy_ratio']:.2f}  "
              f"curtain px={res['pct_curtain_pixels']:.2f}%")
    except Exception as exc:
        print(f"[{k}/{n}] {os.path.basename(path):40s} FAILED: {exc}")
        rows.append({"file": os.path.basename(path), "path": path,
                     "curtaining_std": np.nan, "wedge_energy_ratio": np.nan,
                     "pct_curtain_pixels": np.nan, "wedge_angle_deg": WEDGE_ANGLE_DEG})

df = pd.DataFrame(rows)
out_csv = os.path.join(OUT_DIR, CSV_NAME)
df.to_csv(out_csv, index=False)
print(f"\nSaved {len(df)} row(s) to: {out_csv}")
if SAVE_PICTURES:
    print(f"Pictures saved under: {PIC_DIR}")
df